In [ ]:
import io
import numpy as np
import pandas as pd

# Create dummy CSV data with no header and 5 columns
csv_data = """A,10,20,30,40
B,15,25,35,45
C,12,22,32,42"""

# Load with custom column names and set the first column ('col_0') as the index
df1 = pd.read_csv(
    io.StringIO(csv_data),
    header=None,
    names=["col_0", "col_1", "col_2", "col_3", "col_4"],
    index_col=0,
)

print(df1)

       col_1  col_2  col_3  col_4
col_0                            
A         10     20     30     40
B         15     25     35     45
C         12     22     32     42


In [ ]:
# Sample DataFrame containing missing values
df2 = pd.DataFrame(
    {"A": [1, 2, np.nan, 4], "B": [10, np.nan, 30, 40], "C": [100, 200, 300, 400]}
)

# Export using ';' delimiter and replacing NaN with -999
df2.to_csv("data_missing.csv", sep=";", na_rep="-999", index=False)

# Read it back treating -999 as NaN
df2_read = pd.read_csv("data_missing.csv", sep=";", na_values="-999")

print(df2_read)

     A     B    C
0  1.0  10.0  100
1  2.0   NaN  200
2  NaN  30.0  300
3  4.0  40.0  400


In [ ]:
# First, generate a sample ex6.csv file to demonstrate
np.random.seed(42)
sample_ex6 = pd.DataFrame(
    {
        "key": np.random.choice(["A", "B", "C"], size=5000),
        "one": np.random.randn(5000),
    }
)
sample_ex6.to_csv("ex6.csv", index=False)

# Chunk processing solution
chunker = pd.read_csv("ex6.csv", chunksize=2000)

total_sum = pd.Series(dtype=float)
total_count = pd.Series(dtype=float)

for chunk in chunker:
    # Sum and count per key for each chunk
    chunk_sum = chunk.groupby("key")["one"].sum()
    chunk_count = chunk.groupby("key")["one"].count()

    total_sum = total_sum.add(chunk_sum, fill_value=0)
    total_count = total_count.add(chunk_count, fill_value=0)

# Overall mean per key across all chunks
mean_per_key = total_sum / total_count

print(mean_per_key)

key
A   -0.003866
B    0.038389
C   -0.019491
dtype: float64


In [ ]:
df4 = pd.DataFrame({"A": [1, 2, 3], "B": ["x", "y", "z"]})

# Export to JSON string using records orientation
json_data = df4.to_json(orient="records")

# Read JSON back into a DataFrame
df4_read = pd.read_json(io.StringIO(json_data), orient="records")

# Verify equality
are_equal = df4.equals(df4_read)

print("DataFrames are equal:", are_equal)

DataFrames are equal: True


In [ ]:
html_data = """
<table>
    <tr>
        <th>Name</th>
        <th>Age</th>
        <th>Score</th>
    </tr>
    <tr>
        <td>Alice</td>
        <td>24</td>
        <td>88.5</td>
    </tr>
    <tr>
        <td>Bob</td>
        <td>30</td>
        <td>92.0</td>
    </tr>
</table>
"""

# pd.read_html returns a list of DataFrames
tables = pd.read_html(io.StringIO(html_data))
df5 = tables[0]

print(df5)

    Name  Age  Score
0  Alice   24   88.5
1    Bob   30   92.0


In [6]:
import json
import pandas as pd

# Provided JSON string
t = (
    '{"shop": "Fresh Mart", "open": true, "owner": null, "items": ['
    '{"name": "Rice", "price": 60, "qty": 10}, '
    '{"name": "Oil", "price": 150, "qty": 4}, '
    '{"name": "Tea", "price": 120, "qty": 6}]}'
)

# ==========================================
# Practical 1: JSON basics
# ==========================================

# 1. Load t into a Python object and print its type
data = json.loads(t)
print("1. Type of loaded object:", type(data))

# 2. Print the name of the second item and the number of items
second_item_name = data["items"][1]["name"]
item_count = len(data["items"])
print(f"2. Second item: {second_item_name}, Number of items: {item_count}")

# 3. Print the value of owner. What Python value is it?
owner_val = data["owner"]
print(f"3. Owner value: {owner_val} (Type: {type(owner_val).__name__})")
# JSON 'null' becomes Python 'None'

# 4. Convert the object back to JSON text with indentation 2
json_text = json.dumps(data, indent=2)
print("4. JSON with indentation 2:\n", json_text)


# ==========================================
# Practical 2: JSON to DataFrame
# ==========================================

# 1. Make a DataFrame
df = pd.DataFrame(data["items"])
print("\n--- DataFrame ---")
print(df)

# 2. Add a column total = price x qty
df["total"] = df["price"] * df["qty"]
print("\n--- DataFrame with 'total' Column ---")
print(df)

# 3. Print the grand total and the name of the item with the highest total
grand_total = df["total"].sum()
max_total_item = df.loc[df["total"].idxmax(), "name"]

print(f"\nGrand Total: {grand_total}")
print(f"Item with highest total: {max_total_item}")

1. Type of loaded object: <class 'dict'>
2. Second item: Oil, Number of items: 3
3. Owner value: None (Type: NoneType)
4. JSON with indentation 2:
 {
  "shop": "Fresh Mart",
  "open": true,
  "owner": null,
  "items": [
    {
      "name": "Rice",
      "price": 60,
      "qty": 10
    },
    {
      "name": "Oil",
      "price": 150,
      "qty": 4
    },
    {
      "name": "Tea",
      "price": 120,
      "qty": 6
    }
  ]
}

--- DataFrame ---
   name  price  qty
0  Rice     60   10
1   Oil    150    4
2   Tea    120    6

--- DataFrame with 'total' Column ---
   name  price  qty  total
0  Rice     60   10    600
1   Oil    150    4    600
2   Tea    120    6    720

Grand Total: 1920
Item with highest total: Tea
